In [0]:
dbutils.fs.put("s3://cc3105-au-869227452861-us-east-2-an//equipo4/prueba_escritura.txt", "hola", True)

In [0]:
import os
from pyspark.sql import functions as F, Window

CATALOGO = "equipo4"              # el mismo que en Lector e Ingesta_MAGA

# --- Reglas de limpieza ---
RANGO_VALIDO = (20.0, 50.0)       # Q/galón posible
TOLERANCIA_Q = 3.0                # diferencia máxima aceptada entre la lectura y el promedio oficial del mes
OFICIALES    = ["diesel", "regular", "super"]
TODOS        = OFICIALES + ["v_power"]

T_BRONZE_LECT = f"{CATALOGO}.bronze.lecturas_raw"
T_BRONZE_MAGA = f"{CATALOGO}.bronze.precios_maga_raw"
T_SILVER_OFIC = f"{CATALOGO}.silver.precios_oficiales_mensuales"
T_SILVER_LECT = f"{CATALOGO}.silver.lecturas_fotos"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.silver")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.silver.archivos")
CORRECCIONES = f"/Volumes/{CATALOGO}/silver/archivos/correcciones.csv"   # opcional: archivo,columna,valor

for t in (T_BRONZE_LECT, T_BRONZE_MAGA):
    print(f"{t}: {spark.table(t).count()} filas" if spark.catalog.tableExists(t) else f"⚠️  Falta {t}: corre su notebook primero")

## Preios mensuales

In [0]:
maga = (spark.table(T_BRONZE_MAGA)
        .withColumn("anio_anuario", F.regexp_extract("archivo_fuente", r"(20\d{2})", 1).cast("int"))
        .filter(F.col("precio_q_galon").between(5, 100)))           # descarta números absurdos del PDF
w = Window.partitionBy("combustible", "anio", "mes").orderBy(F.desc("anio_anuario"), F.desc("extraido_en"))
oficial = (maga.withColumn("_n", F.row_number().over(w)).filter("_n = 1")
               .select("combustible", "anio", "mes",
                       F.make_date("anio", "mes", F.lit(1)).alias("mes_inicio"),
                       F.round("precio_q_galon", 2).alias("precio_oficial"),
                       F.col("archivo_fuente").alias("fuente"))
               .withColumn("procesado_en", F.current_timestamp()))

oficial.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_SILVER_OFIC)
print(f"Silver: {T_SILVER_OFIC}  ({oficial.count()} filas)")
display(spark.table(T_SILVER_OFIC).groupBy("combustible")
        .agg(F.min("mes_inicio").alias("desde"), F.max("mes_inicio").alias("hasta"), F.count("*").alias("meses")))

## 2. Lecturas de fotos limpias

In [0]:
# Lecturas: una fila por foto (la más reciente si se procesó varias veces)
w = Window.partitionBy("archivo").orderBy(F.desc("procesado_en"))
lect = (spark.table(T_BRONZE_LECT)
        .withColumn("_n", F.row_number().over(w)).filter("_n = 1").drop("_n")
        .withColumn("fecha", F.to_date("fecha_hora"))
        .withColumn("anio", F.year("fecha_hora"))
        .withColumn("mes", F.month("fecha_hora")))
for c in TODOS:
    if c not in lect.columns:                     # por si alguna columna no vino en bronze
        lect = lect.withColumn(c, F.lit(None).cast("double"))
    lect = lect.withColumn(c, F.when(F.isnan(c), None).otherwise(F.col(c)).cast("double"))   # NaN -> vacío

# Precio oficial del mes de la foto, en columnas: oficial_diesel, oficial_regular, oficial_super
ofic_ancho = (spark.table(T_SILVER_OFIC).groupBy("anio", "mes")
              .pivot("combustible", OFICIALES).agg(F.first("precio_oficial")))
for c in OFICIALES:
    ofic_ancho = ofic_ancho.withColumnRenamed(c, f"oficial_{c}")
lect = lect.join(ofic_ancho, ["anio", "mes"], "left")

# Correcciones manuales (opcional), en columnas: corr_diesel, ..., corr_v_power
if os.path.exists(CORRECCIONES):
    corr = (spark.read.csv(CORRECCIONES, header=True)
            .select(F.trim("archivo").alias("archivo"), F.lower(F.trim("columna")).alias("columna"),
                    F.col("valor").cast("double").alias("valor"))
            .filter(F.col("columna").isin(TODOS)))
    print(f"Correcciones: {corr.count()}")
    corr = corr.groupBy("archivo").pivot("columna", TODOS).agg(F.last("valor"))
    for c in TODOS:
        corr = corr.withColumnRenamed(c, f"corr_{c}")
    lect = lect.join(corr, "archivo", "left")
else:
    print("Sin correcciones.csv (opcional)")
for c in TODOS:
    if f"corr_{c}" not in lect.columns:
        lect = lect.withColumn(f"corr_{c}", F.lit(None).cast("double"))

In [0]:
def limpiar(df, c, con_oficial):
    df = df.withColumnRenamed(c, f"{c}_leido")          # lo que leyó la foto, intacto
    leido, corr = F.col(f"{c}_leido"), F.col(f"corr_{c}")
    en_rango = leido.between(*RANGO_VALIDO)
    if con_oficial:
        ofi = F.col(f"oficial_{c}")
        cerca = F.abs(leido - ofi) <= TOLERANCIA_Q
        final = (F.when(corr.isNotNull(), corr)
                  .when(en_rango & ofi.isNotNull() & cerca, leido)
                  .when(en_rango & ofi.isNull(), leido)
                  .when(ofi.isNotNull(), ofi))
        origen = (F.when(corr.isNotNull(), "corregido_manual")
                   .when(en_rango & ofi.isNotNull() & cerca, "foto")
                   .when(en_rango & ofi.isNull(), "foto_sin_referencia")
                   .when(ofi.isNotNull(), "oficial_maga"))
        motivo = (F.when(corr.isNotNull(), None)
                   .when(leido.isNull(), "no se leyó")
                   .when(~en_rango, "fuera de rango")
                   .when(ofi.isNotNull() & ~cerca, F.concat(F.lit("lejos del oficial ("),
                                                           F.format_number(leido - ofi, 2), F.lit(")"))))
    else:
        final  = F.when(corr.isNotNull(), corr).when(en_rango, leido)
        origen = F.when(corr.isNotNull(), "corregido_manual").when(en_rango, "foto_sin_referencia")
        motivo = (F.when(corr.isNotNull(), None).when(leido.isNull(), "no se leyó")
                   .when(~en_rango, "fuera de rango"))
    return (df.withColumn(c, F.round(final, 2))
              .withColumn(f"{c}_origen", origen)
              .withColumn(f"{c}_descartado", motivo))

for c in TODOS:
    lect = limpiar(lect, c, c in OFICIALES)

origenes = [F.col(f"{c}_origen") for c in TODOS]
lect = (lect
        .withColumn("precios_de_foto",   sum(F.when(o.isin("foto", "foto_sin_referencia"), 1).otherwise(0) for o in origenes))
        .withColumn("precios_corregidos", sum(F.when(o == "corregido_manual", 1).otherwise(0) for o in origenes))
        .withColumn("precios_rellenados", sum(F.when(o == "oficial_maga", 1).otherwise(0) for o in origenes))
        .withColumn("precios_vacios",     sum(F.when(o.isNull(), 1).otherwise(0) for o in origenes))
        .withColumn("procesado_en", F.current_timestamp()))

columnas = (["archivo", "carpeta", "fecha_hora", "fecha", "anio", "mes", "dispositivo", "latitud", "longitud", "altitud_m"]
            + [x for c in TODOS for x in (c, f"{c}_origen", f"{c}_leido", f"{c}_descartado")]
            + [f"oficial_{c}" for c in OFICIALES]
            + ["confianza", "venta_total_q", "venta_galones",
               "precios_de_foto", "precios_corregidos", "precios_rellenados", "precios_vacios", "procesado_en"])
silver = lect.select(*[c for c in columnas if c in lect.columns])

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_SILVER_LECT)
print(f"Silver: {T_SILVER_LECT}  ({silver.count()} filas)")

## 3. Revisión

In [0]:
s = spark.table(T_SILVER_LECT)
# ¿De dónde salió cada precio?
display(s.select(F.explode(F.array(*[F.struct(F.lit(c).alias("combustible"), F.col(f"{c}_origen").alias("origen"))
                                     for c in TODOS])).alias("x"))
         .groupBy("x.combustible", "x.origen").count().orderBy("combustible", "origen"))
# Fotos con lecturas descartadas (candidatas a correcciones.csv)
display(s.filter(" OR ".join(f"{c}_descartado IS NOT NULL" for c in TODOS))
         .select("archivo", "fecha", *[x for c in TODOS for x in (f"{c}_leido", c, f"{c}_descartado")])
         .orderBy("fecha"))

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

COLORES = {"diesel": "#333333", "regular": "#E0B000", "super": "#2E8B57", "v_power": "#C8102E"}

serie = (spark.table(T_SILVER_LECT)
         .select("fecha_hora", *TODOS, *[f"{c}_origen" for c in TODOS]).toPandas()
         .dropna(subset=["fecha_hora"]).sort_values("fecha_hora"))
ofi = spark.table(T_SILVER_OFIC).select("combustible", "mes_inicio", "precio_oficial").toPandas()
ofi["mes_inicio"] = pd.to_datetime(ofi["mes_inicio"])

if len(serie):
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for c in TODOS:
        d = serie[serie[c].notna()]
        if d.empty:
            continue
        rellenado = d[f"{c}_origen"] == "oficial_maga"
        ax.plot(d["fecha_hora"], d[c], color=COLORES[c], lw=1.2)
        ax.scatter(d.loc[~rellenado, "fecha_hora"], d.loc[~rellenado, c], color=COLORES[c], s=24, zorder=3)
        ax.scatter(d.loc[rellenado, "fecha_hora"], d.loc[rellenado, c], facecolors="white",
                   edgecolors=COLORES[c], s=24, zorder=3)

    # Promedio oficial del mes (MAGA), solo en el periodo que cubren las fotos
    desde = serie["fecha_hora"].min() - pd.offsets.MonthBegin(1)
    hasta = serie["fecha_hora"].max()
    for c in OFICIALES:
        o = ofi[(ofi["combustible"] == c) & ofi["mes_inicio"].between(desde, hasta)].sort_values("mes_inicio")
        if len(o):
            ax.step(o["mes_inicio"], o["precio_oficial"], where="post", color=COLORES[c], ls="--", lw=0.9, alpha=0.55)

    presentes = [c for c in TODOS if serie[c].notna().any()]
    leyenda = [Line2D([], [], color=COLORES[c], lw=2, label=c) for c in presentes] + [
        Line2D([], [], ls="", marker="o", color="gray", label="leído en la foto / corregido"),
        Line2D([], [], ls="", marker="o", markerfacecolor="white", color="gray", label="rellenado con oficial"),
        Line2D([], [], ls="--", color="gray", label="promedio oficial del mes (MAGA)")]
    ax.legend(handles=leyenda, loc="upper center", bbox_to_anchor=(0.5, -0.07), fontsize=8,
              ncol=len(leyenda), frameon=False)
    ax.set_title("Precio por galón"); ax.set_ylabel("Quetzales"); ax.set_xlabel("")
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No hay fotos con fecha para graficar")
